**IMPORTANT:**

You need to run the FastAPI app for connecting to the endpoints!

# Test `search/mal` and `save/search-results` endpoints

In [ ]:
import asyncio
import httpx

BASE_URL = "http://localhost:8000"

async def test_search_and_save(query: str):
    # Modify the read time for bigger anime franchises (currently 2 min)
    timeout = httpx.Timeout(connect=5.0, read=120.0, write=10.0, pool=5.0)

    async with httpx.AsyncClient(timeout=timeout) as client:
        # 1. Call the /search/mal endpoint
        search_response = await client.get(f"{BASE_URL}/search/mal", params={"query": query})
        print("Search status:", search_response.status_code)

        if search_response.status_code != 200:
            print("Search failed:", search_response.text)
            return

        search_results = search_response.json()
        print(f"Got {len(search_results)} results")

        # 2. Call the /save/search-results endpoint with the search results
        save_response = await client.post(
            f"{BASE_URL}/save/search-results",
            json=search_results
        )
        print("Save status:", save_response.status_code)
        print("Save response:", save_response)

# Run the tests
await test_search_and_save("My Hero")

# Test `search/media` endpoint

In [ ]:
import asyncio
import httpx

BASE_URL = "http://localhost:8000"

async def test_search_media(query: str, relation_type: str = None, genre_name: list[str] = None, search_type: str = None):
    # Timeout settings for larger or slower responses
    timeout = httpx.Timeout(connect=5.0, read=20.0, write=10.0, pool=5.0)

    params = {"query": query}
    if relation_type:
        params["relation_type"] = relation_type
    if genre_name:
        params["genre_name"] = genre_name
    if search_type:
        params["search_type"] = search_type

    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.get(f"{BASE_URL}/search/media", params=params)
        print("Search/media status:", response.status_code)

        if response.status_code != 200:
            print("Search/media failed:", response.text)
            return

        results = response.json()
        print(f"Search/media returned {len(results)} result(s)")
        print("Sample result:", results[0] if results else "No results")

# Run the test
await test_search_media("My Hero", relation_type="main", genre_name=["Action", "School"], search_type="title") # Search by title
await test_search_media("Trainings arc", genre_name=["Action", "School"], search_type="description") # Search by description
await test_search_media("", relation_type="main", genre_name=["Action", "School"]) # Search with empty query